In [11]:
%load_ext autoreload
%autoreload 2

import sys

sys.path.insert(0,"..")
from src.vocab.filtering import is_plausible_number_token, is_plausible_string_token, is_plausible_boolean_token, build_plausible_vocab, is_plausible_token_name
from llm_sdk import Small_LLM_Model
from src.vocab.loader import loader_vocab
from src.fsm.value_matcher import generate_boolean_value, generate_number_value
from src.generation.decoding import decode_vocab
from src.fsm.value_matcher import generate_string_value
from src.fsm.name_matcher import name_matcher
from src.models.load_init import load_function_catalog, what_function_are_calling, build_function_definition
from src.models.load_init import FunctionCatalog

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
catalog = load_function_catalog("../data/input/functions_definition.json")
descriptions = build_function_definition(catalog)
initial_context = f"Available functions:\n{descriptions}\n\n"

In [13]:
prompts = "Replace all numbers in \"Hello 34 I'm 233 years old\" with NUMBERS"
full_text_json = f'{{"prompt": "{prompts}", "name": "'
full_text_context = initial_context + full_text_json
model = Small_LLM_Model()
dico_inverse = loader_vocab(model)
dico_decode = decode_vocab(dico_inverse, model)
allowed_characters = {c for w in catalog.functions for c in w.name} 
plausible_number = build_plausible_vocab(dico_inverse, is_plausible_number_token)
plausible_string = build_plausible_vocab(dico_decode, is_plausible_string_token)
plausible_boolean = build_plausible_vocab(dico_inverse, is_plausible_boolean_token)
plausible_token_name = build_plausible_vocab(dico_inverse, lambda x: is_plausible_token_name(x, allowed_characters))
noms_valide = [w.name for w in catalog.functions]


In [14]:
def generate_function_call(
    prompt: str,
    catalog: FunctionCatalog,
    model: Small_LLM_Model,
    initial_context: str,
    noms_valide: list[str],
    plausible_token_name: dict[int, str],
    plausible_number: dict[int, str],
    plausible_string: dict[int, str],
    plausible_boolean: dict[int, str],
) -> str:
    """
    Generate a function call based on the provided prompt and catalog.

    Args:
        prompt (str): The input prompt for the model.
        catalog (FunctionCatalog): The catalog of available functions.
        model (Small_LLM_Model): The language model to use for generation.
        initial_context (str): The initial context to provide to the model.
        noms_valide (list[str]): A list of valid function names.
        plausible_token_name (dict[int, str]): A mapping of token IDs to plausible token names.
        plausible_number (dict[int, str]): A mapping of token IDs to plausible number tokens.
        plausible_string (dict[int, str]): A mapping of token IDs to plausible string tokens.
        plausible_boolean (dict[int, str]): A mapping of token IDs to plausible boolean tokens.
    
    Returns:
        str: The generated function call as a string.
    """
    full_text_json = f'{{"prompt": "{prompt}", "name": "'
    full_text_context = initial_context + full_text_json
    full_text = name_matcher(full_text_context, noms_valide, plausible_token_name, model)
    function_name = full_text[len(initial_context + full_text_json):-1]
    functions = what_function_are_calling(function_name, catalog)
    full_text += ', "parameters": {'
    for index, (key, value) in enumerate(functions.parameters.items()):
        is_last = index == len(functions.parameters) - 1
        full_text += f'"{key}": '
        if value.type == "number":
            full_text += generate_number_value(full_text, plausible_number, model)
        elif value.type == "string":
            full_text += '"'
            result = generate_string_value(full_text, plausible_string, model) + '"'
            full_text += result
        elif value.type == "boolean":
            full_text += generate_boolean_value(full_text, plausible_boolean, model)
        if not is_last:
            full_text += ", "
        else:
            full_text += "}}"
    return full_text[len(initial_context):]


In [15]:
prompt_liste = [
    "What is the sum of 2 and 3?",
    "What is the sum of 265 and 345?",
    "Greet shrek",
    "Greet john",
    "Reverse the string 'hello'",
    "Reverse the string 'world'",
    "What is the square root of 16?",
    "Calculate the square root of 144",
    "Replace all numbers in \"Hello 34 I'm 233 years old\" with NUMBERS",
    "Replace all vowels in 'Programming is fun' with asterisks",
    "Substitute the word 'cat' with 'dog' in 'The cat sat on the mat with another cat'"
]
for prompt in prompt_liste:
    result = generate_function_call(
        prompt,
        catalog,
        model,
        initial_context,
        noms_valide,
        plausible_token_name,
        plausible_number,
        plausible_string,
        plausible_boolean,
    )
    print(result)

{"prompt": "What is the sum of 2 and 3?", "name": "fn_add_numbers", "parameters": {"a": 2, "b": 3}}
{"prompt": "What is the sum of 265 and 345?", "name": "fn_add_numbers", "parameters": {"a": 265, "b": 345}}
{"prompt": "Greet shrek", "name": "fn_greet", "parameters": {"name": "shrek"}}
{"prompt": "Greet john", "name": "fn_greet", "parameters": {"name": "john"}}
{"prompt": "Reverse the string 'hello'", "name": "fn_reverse_string", "parameters": {"s": "hello"}}
{"prompt": "Reverse the string 'world'", "name": "fn_reverse_string", "parameters": {"s": "world"}}
{"prompt": "What is the square root of 16?", "name": "fn_get_square_root", "parameters": {"a": 16}}
{"prompt": "Calculate the square root of 144", "name": "fn_get_square_root", "parameters": {"a": 144}}
{"prompt": "Replace all numbers in "Hello 34 I'm 233 years old" with NUMBERS", "name": "fn_substitute_string_with_regex", "parameters": {"source_string": "Hello 34 I'm 233 years old", "regex": "([0-9]+", "replacement": "NUMBERS"}}
{"